# 00 · NVIDIA Probe: what the build.nvidia.com key reaches, for X3D

Turns ATLAS §2 into receipts: the catalog, X3D-writing scores per model, whether USD Code knows the OpenUSD 26.03 splat schema, and (opt-in) TRELLIS image → glb.
**Key:** Colab Secret `NVIDIA_API_KEY` (or a hidden prompt). Receipts store only its sha256 prefix. **Budget:** ~6–10 requests; TRELLIS spends credit.

In [ ]:
# ── Options ──────────────────────────────────────────────────────────────
VERBOSE = True   # stream progress and important log lines while tools run
VISUAL  = True   # show charts and frames as soon as each exists
REPO    = "hagopjay/x3d.nVidia"
REF     = "main"

RUN_CHAT    = True
RUN_TRELLIS = False          # spends credit: image -> glb
RPM         = 36             # x3d.lab shares this key: lower it if the Duel is running

In [ ]:
# ── Setup: clone (private-repo safe), install. No token is ever printed or saved. ──
import os, sys, subprocess, base64, pathlib, time
def sh(cmd, env=None):
    if VERBOSE: print("$", cmd if isinstance(cmd, str) else " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, shell=isinstance(cmd, str), env=env, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    last = 0.0
    for line in p.stdout:
        if VERBOSE and (time.time() - last > 0.2 or "rror" in line):
            print("  " + line.rstrip()[:240], flush=True); last = time.time()
    if p.wait(): raise RuntimeError(f"exit {p.returncode}: {cmd}")

env = dict(os.environ)
try:
    from google.colab import userdata
    tok = userdata.get("GITHUB_TOKEN")
except Exception:
    tok = os.environ.get("GITHUB_TOKEN")
if tok:  # header scoped to this repo; never in the URL
    env.update(GIT_CONFIG_COUNT="1",
               GIT_CONFIG_KEY_0=f"http.https://github.com/{REPO}.extraheader",
               GIT_CONFIG_VALUE_0="AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{tok}".encode()).decode())
root = pathlib.Path("/content/x3d.nVidia") if pathlib.Path("/content").exists() else pathlib.Path("x3d.nVidia").resolve()
if not (root / ".git").exists():
    sh(["git", "clone", "--depth", "1", "-b", REF, f"https://github.com/{REPO}", str(root)], env=env)
else:
    sh(["git", "-C", str(root), "pull", "--ff-only"], env=env)
sh([sys.executable, "-m", "pip", "install", "-q", "-e", f"{root}[dev]"])
sys.path.insert(0, str(root / "src")); os.chdir(root)
print("x3d.nVidia at", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
# ── Probe (streams as it goes) ──
import json
from x3dnv import probe
args = ["--out", "receipts", "--rpm", str(RPM)]
if not RUN_CHAT: args.append("--skip-chat")
if RUN_TRELLIS: args += ["--trellis", "docs/img/trellis_input.png"]
rec = probe.main(args)

In [ ]:
# ── Visual: catalog groups, scores, latency ──
if VISUAL:
    import matplotlib.pyplot as plt
    cat = rec.get("catalog") or {}
    if cat.get("groups"):
        g = cat["groups"]; plt.figure(figsize=(7, 2.6))
        plt.barh(list(g), [len(v) for v in g.values()]); plt.title(f"catalog: {cat['count']} model ids"); plt.show()
        for k, v in g.items(): print(f"{k:9s}", ", ".join(v[:12]) + (" …" if len(v) > 12 else ""))
    rows = [(r["model"].split("/")[-1], r.get("check", {}).get("score"), r["seconds"]) for r in rec["x3d"]]
    if rows:
        fig, ax = plt.subplots(1, 2, figsize=(10, 2.8))
        ax[0].barh([r[0] for r in rows], [r[1] or 0 for r in rows]); ax[0].set_xlim(0, 5); ax[0].set_title("X3D triage score (0–5)")
        ax[1].barh([r[0] for r in rows], [r[2] or 0 for r in rows]); ax[1].set_title("seconds per request"); plt.tight_layout(); plt.show()
    for r in rec["usd"]:
        print(r["model"], "markers", sum((r.get("markers") or {}).values()), "/ 5", "UsdVol" if r.get("mentions_UsdVol") else "")
    if rec.get("trellis"):
        print(json.dumps(rec["trellis"], indent=1)[:2000])

**Next:** commit `receipts/probe-*.json` in a kit and update ATLAS §2 (🔎 → ✅/❌). If TRELLIS returned a glb, open it in X_ITE with `Inline { url "trellis-….glb" }`; that's Q2's first frame.